# PINN Nozzle Solver — Build Your Own Physics-Informed Neural Network From Scratch

<a href="https://colab.research.google.com/github/harsh147-github/PINNs---Bernoulli/blob/main/notebooks/PINN_Nozzle_Solver.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

**Who this is for:** an engineer with a solid fluids/numerics background (you already know what continuity and Bernoulli's theorem are, what an ODE solver does, and what "converged" means) but **zero prior exposure to machine learning or neural networks.** By the end of this notebook, "PINN" will no longer be a black-box term — you will have seen, in running code, every step from "here are the governing equations" to "here is a trained network answering flow queries in microseconds," and you will have a concrete checklist for applying the same method to a different problem of your own.

**What we're building:** a neural network that predicts water velocity and pressure through a converging–diverging duct (a venturi), trained using **only the governing physics equations as the training signal — zero CFD simulation data, zero labeled examples.** The throat diameter and inlet velocity are network inputs, so one trained model answers "what happens if I narrow the throat, or change the inlet speed?" for the *entire* design space at once, in a single batched evaluation.

**How this notebook is organized** (deliberately *not* in source-code file order — in the order a total beginner needs to meet these ideas):

| Part | What you'll learn | Runs live? |
|---|---|---|
| 0 | What a PINN even is, on a trivial 1-line ODE everyone already knows the answer to | ✅ trains a toy PINN |
| 1 | The real problem: nozzle physics, re-derived slowly, symbol by symbol | plots only |
| 2 | Equations → code → loss function, mapped line by line against this repo's actual `src/` code | code walkthrough |
| 3 | How training and fast inference actually work | ✅ trains the real nozzle PINN |
| 4 | Validating against an independent classical solver — the methodology, not just the numbers | ✅ runs an RK45 solve |
| 5 | The inference-speed argument, measured honestly | ✅ runs a timing benchmark |
| 6 | A checklist for building your own PINN for a different problem | reference only |

> **Honesty up front, so this doesn't read as a sales pitch:** PINNs are not, in general, a drop-in replacement for mature CFD tooling. This notebook is upfront in Part 4 about exactly what its cross-validation does and does not prove, and in Part 5 about exactly what its speed numbers do and do not generalize to. Read those caveats — they're load-bearing, not disclaimers of convenience.


## Setup

Clones the repo (skipped if you're already running this notebook from inside a local checkout) and installs dependencies.

In [ ]:
import os, subprocess

IN_REPO = os.path.exists("src/geometry.py")
if not IN_REPO:
    if not os.path.exists("PINNs---Bernoulli"):
        subprocess.run(["git", "clone", "--quiet",
                         "https://github.com/harsh147-github/PINNs---Bernoulli.git"], check=True)
    os.chdir("PINNs---Bernoulli")

print("Working directory:", os.getcwd())


In [ ]:
%pip install -q -r requirements.txt


In [ ]:
import inspect
import math
import time

import numpy as np
import torch
import matplotlib.pyplot as plt
import yaml

torch.manual_seed(1234)
np.random.seed(1234)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "(everything here also runs fine on CPU-only Colab)")


---
# Part 0 — PINNs from absolute zero

Before we touch the nozzle at all, let's learn the *concept* of a Physics-Informed Neural Network on the simplest possible differential equation — one whose answer you already know:

$$\frac{dy}{dx} = -y, \qquad y(0) = 1 \qquad \Longrightarrow \qquad y(x) = e^{-x}$$

**Why this toy problem, and not the nozzle, for the first pass:** the nozzle has geometry, two coupled equations, and a design parameter all at once. This equation has none of that — just one unknown function, one first-order ODE, one boundary condition. Every idea we need (residuals, autodiff, loss functions, training with no data) shows up here in its simplest possible form, isolated from the nozzle's extra complexity. Once you've seen it work here, Part 2 is just "the same four ideas, on two coupled equations instead of one."

### 0.1 The "old way" — how you'd normally solve this

You already know how to do this without any machine learning: pick a numerical ODE integrator (explicit Euler is the simplest; `scipy.integrate.solve_ivp` gives you a much better one, e.g. the RK45 method we'll use again in Part 4) and march forward from the initial condition.

In [ ]:
from scipy.integrate import solve_ivp

def dydx(x, y):
    return -y

sol = solve_ivp(dydx, t_span=(0, 5), y0=[1.0], t_eval=np.linspace(0, 5, 200))

x_exact = np.linspace(0, 5, 200)
y_exact = np.exp(-x_exact)

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(sol.t, sol.y[0], label="solve_ivp (RK45) -- the 'old way'")
ax.plot(x_exact, y_exact, "--", label="exact: $e^{-x}$")
ax.set(xlabel="x", ylabel="y", title="dy/dx = -y, y(0)=1 -- solved the classical way")
ax.legend(); ax.grid(alpha=0.3)
plt.show()
print("This is our anchor: we KNOW the right answer here, both by formula and by a trusted numerical method.")
print("Everything below has to reproduce this same curve -- using a neural network instead, and NO data.")


### 0.2 What a neural network actually is, concretely

Forget every mystical connotation of "neural network." A tiny network that maps one number $x$ to one number $\hat y$ is *nothing more than* a chain of three operations, repeated a few times:

```
x  --[multiply by weights, add a bias]-->  z  --[squash through tanh]-->  a  --[repeat]--> ... --> y_hat
```

Concretely, a network with one hidden layer of $H$ neurons computes:

$$z_i = w_i x + b_i \quad (i = 1 \ldots H), \qquad a_i = \tanh(z_i), \qquad \hat y = \sum_i v_i a_i + c$$

That's it. $H$ "multiply-add-squash" units run side by side on the same input $x$, then their outputs get combined once more into the final number $\hat y$. Every $w_i, b_i, v_i, c$ is a **trainable weight** — a plain number that starts out random and gets nudged during training. A "deep" network just chains more of these layers, each one's output feeding the next one's input (we'll use exactly this — 6 hidden layers — starting in Part 2).

**Why `tanh` and not, say, a jagged step function?** We are about to differentiate this network's output with respect to $x$ — twice, in fact (once to build the physics residual, once more during training's backward pass). `tanh` is smooth everywhere, at every order of derivative. A function with a kink (like ReLU, the default activation almost everywhere else in deep learning) would poison exactly the derivative we need.

Let's build the smallest possible version of this in PyTorch and watch the "multiply, add, squash" chain happen, explicitly:

In [ ]:
import torch.nn as nn

class TinyNet(nn.Module):
    # A hand-built 1 -> 20 -> 20 -> 1 tanh network, to see every step explicitly.

    def __init__(self, n_hidden: int = 20):
        super().__init__()
        self.layer1 = nn.Linear(1, n_hidden)   # x -> z1 = w*x + b  (H weights, H biases)
        self.layer2 = nn.Linear(n_hidden, n_hidden)
        self.layer3 = nn.Linear(n_hidden, 1)   # combine hidden activations -> one output
        self.act = nn.Tanh()                    # the "squash"

    def forward(self, x):
        z1 = self.layer1(x)      # multiply + add (linear combination)
        a1 = self.act(z1)        # squash
        z2 = self.layer2(a1)     # multiply + add, again
        a2 = self.act(z2)        # squash, again
        y_hat = self.layer3(a2)  # final linear combination -> one number out
        return y_hat

toy_net = TinyNet().double()
n_params = sum(p.numel() for p in toy_net.parameters())
print(f"TinyNet has {n_params} trainable numbers (weights + biases), all currently random.")

x_sample = torch.tensor([[0.5]], dtype=torch.float64)
print(f"\nA random, UNTRAINED network's guess at y(0.5): {toy_net(x_sample).item():.4f}")
print(f"The correct answer, e^-0.5, is: {math.exp(-0.5):.4f}  (no reason for these to match yet -- nothing trained)")


### 0.3 The real leap: how do you train this with NO labeled examples?

Ordinary machine learning needs labeled data: pairs $(x_i, y_i)$ where you already know the right answer, and you nudge the weights until $\hat y(x_i) \approx y_i$ for all of them. **We have no such pairs here.** We never generate a single $(x, e^{-x})$ example and never show one to the network. Instead, we exploit the fact that we know the network's output must satisfy a *differential equation*: $dy/dx + y = 0$ everywhere.

To check that on the network's *current guess*, we need $d\hat y/dx$ of that guess — the literal calculus derivative of a function made of matrix multiplies and `tanh` calls. This is where **automatic differentiation (autodiff)** comes in, and it is the one trick that makes any of this possible.

**What autodiff actually is:** as PyTorch runs the forward pass above (`layer1` → `tanh` → `layer2` → `tanh` → `layer3`), it silently records every operation in a *computational graph* — a record of exactly how the output was built from the input, operation by operation. `torch.autograd.grad(y_hat, x)` then walks that graph **backward**, applying the chain rule at each recorded step, and returns the **exact** derivative $d\hat y/dx$ — not a finite-difference approximation, not a numerical estimate with truncation error, the literal calculus answer, to floating-point precision.

Let's prove that to ourselves on something we can check by hand before trusting it on our network: $f(x) = x^3 \Rightarrow f'(x) = 3x^2$.

In [ ]:
x_check = torch.tensor([2.0], dtype=torch.float64, requires_grad=True)
f = x_check ** 3
df_dx = torch.autograd.grad(f, x_check, create_graph=True)[0]

print(f"autograd:  d(x^3)/dx at x=2 -> {df_dx.item()}")
print(f"by hand:   3*x^2   at x=2   -> {3 * 2.0**2}")
print("\nExact match -- not an approximation. Now let's do the same thing to OUR network's output.")


In [ ]:
x_check2 = torch.tensor([[0.5]], dtype=torch.float64, requires_grad=True)
y_hat = toy_net(x_check2)
dy_hat_dx = torch.autograd.grad(y_hat, x_check2, create_graph=True)[0]

# Sanity-check autograd's exact derivative against a manual finite-difference estimate --
# they won't match exactly (finite differences always carry some truncation error), but
# they should agree to several decimal places, which is exactly what we expect.
eps = 1e-6
y_plus = toy_net(x_check2 + eps)
y_minus = toy_net(x_check2 - eps)
fd_estimate = (y_plus - y_minus) / (2 * eps)

print(f"autograd's exact dy_hat/dx at x=0.5:        {dy_hat_dx.item():.8f}")
print(f"central finite-difference estimate (eps={eps}): {fd_estimate.item():.8f}")
print(f"difference: {abs(dy_hat_dx.item() - fd_estimate.item()):.2e}  (tiny -- both are measuring the same slope)")


`create_graph=True` matters here for the same reason it will in Part 2: we're not done differentiating once we have $d\hat y/dx$. We're about to square it into a loss, and then differentiate the loss *again*, with respect to every weight in the network (that second differentiation is `.backward()`, i.e. training). `create_graph=True` keeps the first derivative itself differentiable, so that second pass has a graph to walk.

### 0.4 The loss function, for this toy problem, in full

The differential equation says $dy/dx + y = 0$ everywhere. Whatever the network's current guess $\hat y(x)$ is, we can compute how badly it violates that equation at any point $x$:

$$r(x) = \frac{d\hat y}{dx} + \hat y(x) \qquad \text{("the residual" -- zero only for the true solution)}$$

Squaring and averaging over a batch of points turns this into the one number the optimizer descends:

$$L = \operatorname{mean}\big(r(x)^2\big)$$

**The boundary condition** $y(0)=1$: rather than adding a separate loss penalty for it, we'll use the same "hard boundary condition" trick the nozzle notebook sections use extensively — bake it into the algebra so it can never be violated, no matter what the raw network outputs:

$$\hat y(x) = 1 + x \cdot N(x) \qquad \Longrightarrow \qquad \hat y(0) = 1 + 0 = 1, \ \text{always, for any } N$$

Let's write all of this in code and watch a real PINN converge to $e^{-x}$, with **zero training data** — the only signal is "does your derivative satisfy the ODE.\"

In [ ]:
class ToyPINN(nn.Module):
    def __init__(self, n_hidden: int = 20):
        super().__init__()
        self.net = TinyNet(n_hidden)

    def forward(self, x):
        n_raw = self.net(x)
        y_hat = 1.0 + x * n_raw   # hard boundary condition: y_hat(0) = 1, exactly, always
        return y_hat

def toy_residual(model, x):
    x = x.requires_grad_(True)
    y_hat = model(x)
    dy_dx = torch.autograd.grad(y_hat, x, grad_outputs=torch.ones_like(y_hat), create_graph=True)[0]
    return dy_dx + y_hat   # r(x) = dy_hat/dx + y_hat

def toy_loss(model, x):
    r = toy_residual(model, x)
    return torch.mean(r ** 2)


In [ ]:
torch.manual_seed(0)
toy_model = ToyPINN(n_hidden=20).double()
opt = torch.optim.Adam(toy_model.parameters(), lr=1e-2)

x_plot = torch.linspace(0, 5, 200, dtype=torch.float64).reshape(-1, 1)
y_exact_plot = torch.exp(-x_plot)

snapshots = {}
TOY_EPOCHS = 2000
for epoch in range(1, TOY_EPOCHS + 1):
    # NOTE: these x points are NOT training data with known labels -- they are just
    # locations where we check "does the physics hold here." A fresh random batch
    # every step, exactly like the nozzle's collocation sampling in Part 2.5.
    x_colloc = torch.rand(200, 1, dtype=torch.float64) * 5.0

    opt.zero_grad()
    loss = toy_loss(toy_model, x_colloc)
    loss.backward()
    opt.step()

    if epoch in (1, 10, 50, 200, 500, 2000):
        with torch.no_grad():
            snapshots[epoch] = toy_model(x_plot).clone()
        print(f"epoch {epoch:5d} | loss = {loss.item():.3e}")

print("\nTraining used exactly zero (x, y) example pairs -- only the ODE residual.")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(x_plot.numpy(), y_exact_plot.numpy(), "k--", lw=2, label="exact: $e^{-x}$")
for epoch, y_snap in snapshots.items():
    ax.plot(x_plot.numpy(), y_snap.numpy(), alpha=0.7, label=f"PINN after {epoch} epochs")
ax.set(xlabel="x", ylabel="y", title="A toy PINN converging to $e^{-x}$ with NO training data")
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.show()

final_rel_l2 = (torch.norm(snapshots[TOY_EPOCHS] - y_exact_plot) / torch.norm(y_exact_plot)).item()
print(f"Final relative L2 error vs the exact solution: {final_rel_l2:.2e}")


**Stop and notice what just happened.** We never once told this network what $e^{-x}$ looks like. We wrote down a differential equation, turned "how well does the network's own derivative satisfy that equation" into a number via autodiff, and let gradient descent shrink that number. The curve above converging onto $e^{-x}$ is that process working.

Every remaining part of this notebook is this **exact same four-step recipe** — (1) write a residual using autodiff, (2) square-and-average it into a loss, (3) train with no labeled data, (4) validate against something you trust — applied to two coupled equations (continuity + Bernoulli) instead of one, with a design parameter (throat diameter) added as a second input. Nothing conceptually new happens from here on; only the equations get more interesting.

---
# Part 1 — The real problem: nozzle physics, from scratch

**The setup:** water flows through a duct (a *venturi*) that narrows to a throat and widens back out — picture a length of pipe with a pinch in the middle. We want $V(x)$ (velocity) and $p(x)$ (pressure) at every axial position $x$ along the duct, for *any* throat diameter $D_t$ we might choose.

### 1.1 The four assumptions, and why each is reasonable here

Every equation below rests on four simplifying assumptions. Each one is doing real work — here is what each buys us and what justifies it for *this specific problem* (this repo's actual configuration: water, $\rho=998\,\text{kg/m}^3$, inlet speed $V_{in}=2\,\text{m/s}$, duct length $L=3\,\text{m}$, inlet diameter $D_{in}=0.5\,\text{m}$):

- **Steady** — nothing changes with time; once the flow has settled, velocity and pressure at any *fixed point* in the duct stay constant forever. This eliminates every $\partial/\partial t$ term before we even start.

- **Incompressible** — the fluid's density $\rho$ doesn't change no matter how much the duct squeezes it. The standard way to check whether this is reasonable is the **Mach number**, $M = V/a$ — the ratio of the flow speed to the local speed of sound $a$ in that fluid. It matters because a fluid moving near or above the speed of sound resists further compression very differently than a slow-moving one; textbook practice treats $M \lesssim 0.3$ as safely incompressible. Water's speed of sound is about $a \approx 1480\,\text{m/s}$; at our inlet speed $V_{in}=2\,\text{m/s}$, that's $M = 2/1480 \approx 0.0014$ — three orders of magnitude below the 0.3 threshold. Water at everyday pressures is about as incompressible as fluids get.

- **Inviscid** — we ignore internal friction (viscosity). This one deserves more care than a one-line justification: the **Reynolds number** $Re = \rho V D/\mu$ (the ratio of inertial to viscous forces; $\mu\approx10^{-3}\,\text{Pa·s}$ for water) works out to roughly $Re = 998 \times 2 \times 0.5 / 10^{-3} \approx 10^6$ here — nominally turbulent pipe flow, where viscosity is very much *not* negligible in general. The inviscid assumption we're actually making is narrower than "ignore viscosity everywhere": for a short, smoothly-tapered duct like this one, the friction losses accumulated along the wall are small **compared to the pressure change the area change itself causes** (the effect Bernoulli's theorem describes). It's a comparison of two effects' *sizes*, not a claim that friction is literally zero — and it's the one assumption that turns a genuinely hard problem (the full Navier–Stokes equations) into an easy one (Euler's equations → Bernoulli).

- **Quasi-1D** — we track exactly one velocity number and one pressure number *per cross-section*, as if the fluid moves in flat slugs straight down the axis with no swirling or radial variation. Valid as long as the duct's diameter changes gradually along its length — which ours does by construction (see the shape law below).

### 1.2 The geometry

$$D(x; D_t) = D_{in} + (D_t - D_{in})\sin^2\!\left(\frac{\pi x}{L}\right), \qquad x \in [0, L], \qquad A(x; D_t) = \frac{\pi}{4}D(x;D_t)^2$$

where $x$ [m] is axial position, $D_t$ [m] is the throat diameter (our design parameter), $L$ [m] is the total duct length, $D_{in}$ [m] is the (equal) inlet/outlet diameter, and $A$ [m$^2$] is the local cross-sectional area. Check the endpoints: $\sin(0)=\sin(\pi)=0 \Rightarrow D(0)=D(L)=D_{in}$; at the midpoint $\sin(\pi/2)=1 \Rightarrow D(L/2)=D_t$, the throat, exactly where we want it — and the function is smooth everywhere with zero slope at the throat, so nothing kinks.

In [ ]:
from src import geometry

L, D_in = 3.0, 0.5
x = torch.linspace(0.0, L, 300).reshape(-1, 1)

fig, ax = plt.subplots(figsize=(8, 3))
for Dt in [0.20, 0.30, 0.45]:
    dt = torch.full_like(x, Dt)
    D = geometry.diameter(x, dt, L, D_in)
    ax.plot(x.numpy(), (D / 2).numpy(), label=f"$D_t$={Dt} m")
    ax.plot(x.numpy(), (-D / 2).numpy(), color=ax.lines[-1].get_color())
ax.axvline(L / 2, color="gray", ls=":", lw=1, label="throat ($x=L/2$)")
ax.set(xlabel="x [m]", ylabel="duct half-width [m]", title="Venturi geometry at three throat diameters")
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.show()


### 1.3 Deriving continuity, slowly

Mass has nowhere to disappear to. For steady flow, the mass flow rate $\dot m = \rho A(x) V(x)$ [kg/s] must be the same at every cross-section — whatever goes in at the inlet must come out everywhere downstream:

$$\rho A(x) V(x) = \rho A_{in} V_{in} \qquad \text{(density } \rho \text{ is the same everywhere -- incompressible -- so it cancels)}$$

$$\boxed{A(x)\,V(x) = A_{in}\,V_{in}} \qquad \Longrightarrow \qquad V(x) = \frac{V_{in}A_{in}}{A(x)} \tag{1}$$

### 1.4 Deriving Bernoulli's theorem, slowly (from $F=ma$)

Take a thin slug of fluid of cross-section $A(x)$ and infinitesimal length $dx$, mass $dm = \rho A\, dx$. Newton's second law along the flow direction, with pressure the only force (inviscid — no friction) and steady flow (so $V\,dV/dx$ is the slug's acceleration, from the chain rule $dV/dt = (dV/dx)(dx/dt) = V\,dV/dx$):

$$dm \cdot V\frac{dV}{dx} = -A\frac{dp}{dx} \quad\Longrightarrow\quad \rho A \,dx \cdot V\frac{dV}{dx} = -A\,dp \quad\Longrightarrow\quad \rho V\,dV = -dp$$

Integrate both sides from the inlet to position $x$ (density constant, so it comes out of the integral):

$$\rho\int_{V_{in}}^{V} V\,dV = -\int_{p_{in}}^{p} dp \quad\Longrightarrow\quad \tfrac12\rho\big(V^2 - V_{in}^2\big) = -(p - p_{in})$$

$$\boxed{p(x) + \tfrac12\rho V(x)^2 = p_{in} + \tfrac12\rho V_{in}^2} \tag{2}$$

— pressure trades off against velocity so their combination stays constant. This is exactly Bernoulli's theorem, derived from nothing but $F=ma$ on a fluid slug.

### 1.5 Non-dimensionalization — why the network needs rescaled numbers

Here's a concrete problem: at the inlet, $x=0\,\text m$ but $p=101{,}325\,\text{Pa}$ — six orders of magnitude apart. A freshly-initialized network's weights are small random numbers; asked to output something around $10^5$ from an input around $1$, it would need enormous weights before training even gets started, badly distorting the loss landscape it has to descend. Neural networks train well when everything — inputs *and* outputs — sits around order 1. So we rescale by natural reference values:

$$\tilde x = \frac{x}{L}, \quad \tilde A = \frac{A}{A_{in}}, \quad \tilde D_t = \frac{D_t}{D_{in}}, \quad \tilde V = \frac{V}{V_{in}}, \quad \tilde p = \frac{p - p_{in}}{\tfrac12\rho V_{in}^2}$$

Substituting into Eqs. (1)–(2), every constant ($\rho$, $V_{in}$, $p_{in}$, $L$, $D_{in}$) cancels out completely:

$$\tilde A(\tilde x)\,\tilde V = 1, \qquad \tilde p + \tilde V^2 = 1, \qquad \tilde V(0)=1,\ \ \tilde p(0)=0 \tag{3}$$

and solving these algebraically for $\tilde V$ and $\tilde p$ gives the **closed-form exact answer** we will grade every other method against for the rest of this notebook:

$$\tilde V(\tilde x) = \frac{1}{\tilde A(\tilde x; \tilde D_t)}, \qquad \tilde p(\tilde x) = 1 - \tilde V(\tilde x)^2 \tag{4}$$

Everything the network ever sees or produces from here on is $O(1)$; SI units only reappear at the very last step, converting the trained output back with $\rho$, $V_{in}$, $p_{in}$.

In [ ]:
from src import analytical

xt = torch.linspace(0.0, 1.0, 300).reshape(-1, 1)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for Dt_tilde in [0.4, 0.6, 0.9]:
    dtt = torch.full_like(xt, Dt_tilde)
    v = analytical.velocity_exact_nondim(xt, dtt)
    p = analytical.pressure_exact_nondim(xt, dtt)
    axes[0].plot(xt.numpy(), v.numpy(), label=f"$\\tilde D_t$={Dt_tilde}")
    axes[1].plot(xt.numpy(), p.numpy(), label=f"$\\tilde D_t$={Dt_tilde}")
axes[0].set(xlabel="$\\tilde x$", ylabel="$\\tilde V$", title="Eq. (4): exact velocity")
axes[1].set(xlabel="$\\tilde x$", ylabel="$\\tilde p$", title="Eq. (4): exact pressure")
for ax in axes:
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print("Smaller throat -> higher peak velocity (Eq. 1) -> deeper pressure drop (Eq. 2). This is the whole physics story.")


---
# Part 2 — From the toy example to the real one: equations → code → loss function

This is the section that matters most: for each piece of the puzzle, we'll put the **math** and the **exact corresponding lines of `src/` code** side by side, and narrate the translation explicitly. Every code snippet below is pulled live from the repository with `inspect.getsource` — not retyped — so you're reading precisely what this project trains with.

## 2.1 The geometry law: $D(x;D_t)$ → `geometry.diameter` / `area_nondim`

Recall Part 0.3: autodiff needs a recorded computational graph, which only exists for operations PyTorch knows how to differentiate. The geometry law will get called later with `x.requires_grad_(True)` (Section 2.2, exactly like `x_check2` in Part 0), so it has to be built from `torch.sin`, not plain Python `math.sin` or NumPy — otherwise there's nothing to differentiate through when we need $d\tilde A/d\tilde x$.

In [ ]:
print(inspect.getsource(geometry.diameter))
print(inspect.getsource(geometry.area_nondim))


## 2.2 Continuity + Bernoulli, on paper → `physics.py`, in code

Eq. (3)'s two governing relations, written as **residuals** (differential form — the slope of a quantity that should be constant must be zero):

$$r_{cont} = \frac{d}{d\tilde x}\Big[\tilde A(\tilde x, \tilde D_t)\cdot\hat{\tilde V}\Big], \qquad r_{mom} = \frac{d}{d\tilde x}\Big[\hat{\tilde p} + \hat{\tilde V}^2\Big]$$

Compare this line by line against `physics.residual_continuity`: `flux = area_nondim(xt, dtt) * v` is literally $\tilde A \cdot \hat{\tilde V}$, and `_grad(flux, xt)` is literally $d/d\tilde x[\cdot]$ — the exact same `torch.autograd.grad` mechanism from Part 0.3, just called on the nozzle network's output instead of the toy network's.

In [ ]:
from src import physics

print(inspect.getsource(physics._grad))
print(inspect.getsource(physics.residual_continuity))
print(inspect.getsource(physics.residual_momentum))


**`create_graph=True` inside `_grad`** — exactly the flag we called out in Part 0.3, for exactly the same reason: `losses.py` (Section 2.4) squares this residual, and `train.py` (Part 3) later calls `.backward()` on the total loss to differentiate *again*, this time with respect to the network's weights. Drop this flag and training silently fails at the first `.backward()` call.

## 2.3 Boundary conditions: two ways to enforce $\hat{\tilde V}(0)=1,\ \hat{\tilde p}(0)=0$ — a real design choice

There are genuinely two ways to make a network respect a boundary condition, and it's worth seeing both so the one this repo uses reads as a *choice*, not the only option.

**Option A — a soft penalty loss term.** Add an extra loss term that punishes the network for being far from the correct value at $\tilde x=0$, and let the optimizer trade it off against the physics residuals:

$$L_{bc} = \operatorname{mean}\big[(\hat{\tilde V}(0)-1)^2 + \hat{\tilde p}(0)^2\big]$$

This is exactly `losses.loss_bc_soft` below — it's a real, usable option (this repo even keeps `hard_bc: false` as a config switch that activates it):

In [ ]:
from src import losses

print(inspect.getsource(losses.loss_bc_soft))


The problem with Option A: it's a *suggestion*. Nothing stops the optimizer from leaving a small residual BC violation in exchange for a bigger reduction elsewhere in the loss — early in training, or if $\lambda_{bc}$ is poorly tuned, the boundary condition can end up only approximately satisfied.

**Option B — the hard-BC trick, what this repo actually uses (`hard_bc: true` in `configs/default.yaml`).** Bake the condition directly into the network's output *algebra*, so there is no combination of weights that can violate it:

$$\hat{\tilde V}(\tilde x) = 1 + \tilde x \cdot N_V(\tilde x, \tilde D_t) \ \Rightarrow\ \hat{\tilde V}(0)=1,\ \text{always} \qquad \hat{\tilde p}(\tilde x) = \tilde x \cdot N_p(\tilde x, \tilde D_t) \ \Rightarrow\ \hat{\tilde p}(0)=0,\ \text{always}$$

This is *precisely* the trick Part 0.4's toy PINN used ($\hat y = 1 + x\cdot N(x)$) — the exact same idea, just with two outputs instead of one. Here it is in `networks.PINN.forward`:

In [ ]:
from src import networks

print(inspect.getsource(networks.PINN.forward))


In [ ]:
# Prove the hard-BC ansatz holds EXACTLY, even on an untrained, randomly-initialized network:
demo_model_preview = networks.PINN(n_hidden_layers=6, n_neurons=64, activation="tanh", hard_bc=True).double()
xt0 = torch.zeros(5, 1, dtype=torch.float64)
dtt0 = torch.rand(5, 1, dtype=torch.float64) * 0.5 + 0.4
v0, p0 = demo_model_preview(xt0, dtt0)
print("V~(0) for 5 random throat diameters (should be EXACTLY 1.0):", v0.flatten().tolist())
print("p~(0) for 5 random throat diameters (should be EXACTLY 0.0):", p0.flatten().tolist())
print("\nOption B (hard-wired) is what configs/default.yaml actually uses for this project's results.")


## 2.4 Assembling the loss function — the piece you most need to see explicitly

### Step 1: residual → squared → averaged, for EACH equation separately

$$L_{cont} = \operatorname{mean}\big(r_{cont}^2\big), \qquad L_{mom} = \operatorname{mean}\big(r_{mom}^2\big)$$

- **Squaring** makes a residual of $-0.3$ count exactly as bad as $+0.3$ (direction doesn't matter, only size), and punishes large violations far harder than small ones ($0.3^2=0.09$ vs $0.03^2=0.0009$ — a 10× bigger error costs **100×** more loss).
- **Averaging** turns "thousands of per-point residuals" into the single scalar the optimizer needs.

In [ ]:
print(inspect.getsource(losses.loss_continuity))
print(inspect.getsource(losses.loss_momentum))


In [ ]:
# Build L_cont from scratch, by hand, on the untrained network, to see every step:
xt_demo = torch.rand(2000, 1, dtype=torch.float64)
dtt_demo = torch.rand(2000, 1, dtype=torch.float64) * 0.5 + 0.4

r = physics.residual_continuity(demo_model_preview, xt_demo, dtt_demo)      # step 1: the residual vector
print("residual vector shape:", tuple(r.shape), " (one r_cont per collocation point)")
r_squared = r ** 2                                                          # step 2: square
L_cont_manual = r_squared.mean()                                            # step 3: average -> ONE number
print("L_cont, built by hand:  ", L_cont_manual.item())
print("L_cont, from losses.py: ", losses.loss_continuity(demo_model_preview, xt_demo, dtt_demo).item())


### Step 2: the basic total loss — FIRST, without any refinement

$$L = \lambda_{cont}\,L_{cont} + \lambda_{mom}\,L_{mom} \qquad \text{(start with } \lambda_{cont}=\lambda_{mom}=1\text{, the simplest possible choice)}$$

That's it — this alone is a *complete, trainable* loss function. Part 3 will train exactly this. But before we do, it's worth understanding one wrinkle that shows up with any multi-term physics loss.

### Step 3: the refinement — why fixed $\lambda=1$ isn't always good enough

Continuity and momentum don't naturally produce gradients of the same size. With both weights pinned at 1 forever, the optimizer will happily drive whichever term has the *louder* gradient toward zero while quietly neglecting the other (a "gradient pathology," Wang, Teng & Perdikaris 2021). Let's actually measure this, on our untrained network, right now:

In [ ]:
params = [p for p in demo_model_preview.parameters() if p.requires_grad]
l_cont = losses.loss_continuity(demo_model_preview, xt_demo, dtt_demo)
l_mom = losses.loss_momentum(demo_model_preview, xt_demo, dtt_demo)
g_cont = torch.autograd.grad(l_cont, params, retain_graph=True, allow_unused=True)
g_mom = torch.autograd.grad(l_mom, params, retain_graph=True, allow_unused=True)
mean_abs_grad_cont = torch.cat([g.abs().flatten() for g in g_cont if g is not None]).mean().item()
mean_abs_grad_mom = torch.cat([g.abs().flatten() for g in g_mom if g is not None]).mean().item()
print(f"mean|grad L_cont| = {mean_abs_grad_cont:.3e}")
print(f"mean|grad L_mom|  = {mean_abs_grad_mom:.3e}")
print(f"ratio = {max(mean_abs_grad_cont, mean_abs_grad_mom) / min(mean_abs_grad_cont, mean_abs_grad_mom):.1f}x")
print("\nWith lambda=1 fixed for both, whichever term has the larger gradient above will dominate every optimizer step.")


**The fix — LR-annealing (Wang et al. 2021, Algorithm 1), an enhancement ON TOP of the basic loss, not a replacement for it.** Every `anneal_every` epochs, re-measure each term's gradient *magnitude* and rebalance so no term dominates:

$$\hat\lambda_i = \frac{\max_j|\nabla L_j|}{\operatorname{mean}|\nabla L_i|}, \qquad \lambda_i \leftarrow (1-\alpha)\lambda_i + \alpha\hat\lambda_i \quad(\alpha=0.9)$$

In [ ]:
print(inspect.getsource(losses.LossWeights.total))
print(inspect.getsource(losses.LossWeights.maybe_anneal))


In [ ]:
# Run the annealing update a few times by hand, and watch lambda_cont/lambda_mom
# move to compensate for the exact gradient-magnitude gap we just measured above.
demo_weights_preview = losses.LossWeights(weighting="annealing", anneal_alpha=0.9, anneal_every=1, anneal_warmup=0)
print(f"{'step':>4} | {'lambda_cont':>12} | {'lambda_mom':>12}")
for step in range(5):
    total, parts = demo_weights_preview.total(demo_model_preview, xt_demo, dtt_demo, hard_bc=True)
    demo_weights_preview.maybe_anneal(step, demo_model_preview, parts)  # BEFORE backward -- see the warning below
    print(f"{step:>4} | {demo_weights_preview.values['cont']:>12.3f} | {demo_weights_preview.values['mom']:>12.3f}")


**One documented, real ordering bug, worth learning from:** `train.py` calls `weights.maybe_anneal(...)` **before** `total.backward()`, never after. `maybe_anneal` needs `torch.autograd.grad` on each *individual* loss term, which only works while the computation graph is alive; `.backward()` frees that graph. This exact bug existed in an earlier version of this repository and was fixed — `CLAUDE.md` now states outright: *"Annealing must run BEFORE `total.backward()` — do not regress it."* It's a good first example of the kind of subtle-but-fatal ordering mistake this style of training is prone to.

## 2.5 Collocation sampling — there is no "training data" here, and that's the whole point

If you're used to supervised ML, this is the biggest conceptual departure: **there are no labeled examples anywhere in this pipeline.** We never ran a CFD simulation, never measured a real nozzle, never have a $(x, D_t) \to (V, p)$ answer key. The only thing we sample is *where to check the physics* — points $(\tilde x, \tilde D_t)$ at which the residuals above get evaluated. This is fundamentally different from a dataset: a mislabeled or noisy *data point* would actively corrupt supervised training, but a *collocation point* is never "wrong" — the equation either holds there or it doesn't, and that's determined entirely by the network's own current guess, not by any external label.

`sampling.lhs_collocation` draws these points using **Latin Hypercube Sampling (LHS)**: spread evenly across both axes (no accidental clumping like pure-random sampling, no rigid repeats like a grid), with position $\tilde x$ *and* throat diameter $\tilde D_t$ sampled together every draw — the literal mechanical reason one trained network becomes a **parametric surrogate** across the whole design space, rather than a solver for one fixed geometry.

In [ ]:
from src import sampling

print(inspect.getsource(sampling.lhs_collocation))


In [ ]:
xt_lhs, dtt_lhs = sampling.lhs_collocation(n_points=512 * 8, dtt_min=0.4, dtt_max=0.9, seed=1234)
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(xt_lhs.numpy(), dtt_lhs.numpy(), s=3, alpha=0.4)
ax.set(xlabel="$\\tilde x$", ylabel="$\\tilde D_t$",
       title=f"{xt_lhs.shape[0]} LHS collocation points -- NOT labeled data, just 'where to check physics'")
ax.grid(alpha=0.3)
plt.show()


---
# Part 3 — How training actually works, and how it becomes fast inference

## 3.1 Two optimizers, two jobs, in plain language

Training this network runs in two phases, back to back:

- **Phase 1 — Adam.** Think of Adam as *cautious, adaptive, repeated small steps*: it keeps a running memory of recent gradients for every individual weight and adjusts its own step size per-weight as it goes, which makes it robust to the wildly uneven, chaotic loss landscape a freshly-initialized network starts on. It runs for the bulk of training (20,000 full-batch steps in the real config) and does most of the work of getting the loss down.

- **Phase 2 — L-BFGS.** Think of L-BFGS as *one calculated jump once you're already close*: it's a quasi-Newton method that uses *curvature* information (how the gradient itself is changing, not just its current value) to take far more precise steps than Adam can, squeezing the loss down another one or two orders of magnitude in the final stretch. It's more expensive per step (several forward/backward evaluations per iteration for its internal line search), which is exactly why it only runs for a short final polish (5,000 iterations) rather than the whole run.

**The one real, documented lesson this repo learned the hard way about this two-phase design** (`src/losses.py` / `CLAUDE.md` §5): the loss-weight annealing update (Section 2.4) *must* run before `.backward()` frees the computation graph. This isn't a hypothetical warning — an earlier version of this exact codebase got this ordering wrong, annealing silently broke, and the fix is now a standing rule in this project's build contract. The general lesson: PINN training pipelines have more moving, order-sensitive parts than ordinary supervised training, and it's worth auditing that ordering carefully whenever you build a new one.

## 3.2 Training the real nozzle PINN, live

To keep this notebook fast, we'll run a **short demo** (a few hundred epochs) using the *exact same* functions the full pipeline uses — not a reimplementation. The full, gate-passing protocol (`scripts/run_stage1_bernoulli.py`) runs 20,000 Adam epochs + 5,000 L-BFGS iterations, roughly 15-20 minutes on CPU; see Part 3.4 for where that full run's actual numbers come from.

In [ ]:
with open("configs/default.yaml") as f:
    cfg = yaml.safe_load(f)

torch.manual_seed(cfg["seed"])
demo_model = networks.PINN(
    n_hidden_layers=cfg["network"]["n_hidden_layers"],
    n_neurons=cfg["network"]["n_neurons"],
    activation=cfg["network"]["activation"],
    hard_bc=cfg["network"]["hard_bc"],
).double()

g = cfg["geometry"]
dtt_min, dtt_max = g["dt_min"] / g["D_in"], g["dt_max"] / g["D_in"]

demo_weights = losses.LossWeights(
    weighting=cfg["loss"]["weighting"],
    anneal_alpha=cfg["loss"]["anneal_alpha"],
    anneal_every=cfg["loss"]["anneal_every"],
    anneal_warmup=cfg["loss"]["anneal_warmup"],
    values={"cont": cfg["loss"]["lambda_cont"], "mom": cfg["loss"]["lambda_mom"], "bc": cfg["loss"]["lambda_bc"]},
)

opt = torch.optim.Adam(demo_model.parameters(), lr=cfg["training"]["adam_lr"])
DEMO_EPOCHS = 400  # the real run uses cfg["training"]["adam_epochs"] = 20000

xt_f, dtt_f = sampling.lhs_collocation(
    cfg["training"]["n_collocation_x"] * cfg["training"]["n_dt_values"], dtt_min, dtt_max, cfg["seed"]
)

from src import evaluate

loss_history, snap_epochs, snapshots_v, snapshots_p = [], [], {}, {}
xt_plot = torch.linspace(0, 1, 300, dtype=torch.float64).reshape(-1, 1)
dtt_plot_demo = torch.full_like(xt_plot, 0.6)  # a representative throat for the convergence snapshots

for epoch in range(1, DEMO_EPOCHS + 1):
    opt.zero_grad()
    total, parts = demo_weights.total(demo_model, xt_f, dtt_f, hard_bc=True)
    demo_weights.maybe_anneal(epoch, demo_model, parts)   # BEFORE backward -- Section 2.4's ordering rule
    total.backward()
    opt.step()
    loss_history.append(total.item())

    if epoch in (1, 20, 100, 400):
        with torch.no_grad():
            v_snap, p_snap = demo_model(xt_plot, dtt_plot_demo)
        snap_epochs.append(epoch)
        snapshots_v[epoch] = v_snap.clone()
        snapshots_p[epoch] = p_snap.clone()

    if epoch % 100 == 0:
        print(f"epoch {epoch:5d} | total loss = {total.item():.3e} | "
              f"lambda_cont={demo_weights.values['cont']:.2f}  lambda_mom={demo_weights.values['mom']:.2f}")

print("\ndemo training done -- NOT the full 20,000-epoch run, so the gates checked later will NOT pass yet.")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].semilogy(loss_history)
axes[0].set(xlabel="epoch", ylabel="total loss (log)", title=f"Loss history ({DEMO_EPOCHS} Adam epochs)")
axes[0].grid(alpha=0.3, which="both")

v_exact_snap = analytical.velocity_exact_nondim(xt_plot, dtt_plot_demo)
p_exact_snap = analytical.pressure_exact_nondim(xt_plot, dtt_plot_demo)
for epoch in snap_epochs:
    axes[1].plot(xt_plot.numpy(), snapshots_v[epoch].numpy(), alpha=0.7, label=f"epoch {epoch}")
    axes[2].plot(xt_plot.numpy(), snapshots_p[epoch].numpy(), alpha=0.7, label=f"epoch {epoch}")
axes[1].plot(xt_plot.numpy(), v_exact_snap.numpy(), "k--", lw=2, label="exact")
axes[2].plot(xt_plot.numpy(), p_exact_snap.numpy(), "k--", lw=2, label="exact")
axes[1].set(xlabel="$\\tilde x$", ylabel="$\\tilde V$", title="Velocity converging toward exact ($\\tilde D_t$=0.6)")
axes[2].set(xlabel="$\\tilde x$", ylabel="$\\tilde p$", title="Pressure converging toward exact ($\\tilde D_t$=0.6)")
for ax in axes[1:]:
    ax.legend(fontsize=7); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


## 3.3 Training happens ONCE. Every query after that is just a forward pass.

This is the point the whole speed argument (Part 5) rests on. "Training" — everything in the cell above — updates the network's ~21,000 weights using gradients, an optimizer, and (per Part 0.3/2.2) a computation graph built for `torch.autograd.grad`. **None of that machinery is needed to *use* a trained network.** Querying it is just: plug numbers into `forward()`, get numbers out — ordinary matrix multiplications, `torch.no_grad()`, no backward pass, no optimizer step, nothing to differentiate.

Let's measure exactly how fast that is, for a **batch of 10,000 simultaneous queries answered in one call**:

In [ ]:
from src import benchmark

pinn_timing = benchmark.time_pinn_inference(demo_model, cfg, n_queries=10000)
print(f"10,000 (x, Dt) queries, answered in ONE batched forward pass:")
print(f"  total time:      {pinn_timing['total_seconds']*1000:.2f} ms")
print(f"  time per query:  {pinn_timing['seconds_per_query']*1e6:.3f} microseconds")
print("\nWhy so fast: no autodiff graph is built (torch.no_grad()), so there is nothing for a")
print("backward pass to walk -- this is pure matrix arithmetic, the cheapest thing a network does.")


## 3.4 The real, gate-passing result (for reference — not reproduced live here)

The committed reference run (`scripts/run_stage1_bernoulli.py`, full 20,000 Adam epochs + 5,000 L-BFGS iterations, seed 1234) passes all three acceptance gates (`CLAUDE.md` §5) with roughly a 10× margin:

| Gate | Threshold | Reference run result |
|---|---|---|
| rel-$L^2(\tilde V)$ on held-out $D_t$ | $< 10^{-3}$ | $1.03\times10^{-4}$ |
| rel-$L^2(\tilde p)$ on held-out $D_t$ | $< 10^{-2}$ | $2.33\times10^{-4}$ |
| max Bernoulli-invariant error $|\hat{\tilde p}+\hat{\tilde V}^2-1|$ | $< 1\%$ | $3.4\times10^{-2}\%$ |

We don't reproduce that 15-20 minute run inside this notebook — Section 3.5 (next) checks our own short demo model against the same gates instead, honestly reporting that it does *not* yet pass them, which is the expected and correct outcome after only 400 epochs.

## 3.5 Grading our demo model (`src/evaluate.py`)

$$\text{rel-}L^2 = \frac{\lVert\text{prediction} - \text{exact}\rVert}{\lVert\text{exact}\rVert}$$

measured **only on held-out throat diameters** (0.225, 0.275, 0.325, 0.375, 0.425 m) that never appeared in any `lhs_collocation` draw — a network that merely memorized its training geometries would fail here.

In [ ]:
report = evaluate.full_report(demo_model, cfg)
print(f"rel-L2(V~)        = {report['rel_l2_V']:.3e}   (gate < {cfg['acceptance']['rel_l2_V']:.0e})")
print(f"rel-L2(p~)        = {report['rel_l2_p']:.3e}   (gate < {cfg['acceptance']['rel_l2_p']:.0e})")
print(f"max|p~+V~^2-1|    = {report['total_head_max_err']:.3e}   (gate < {cfg['acceptance']['total_head_tol']:.0e})")
print(f"ALL GATES PASSED? = {report['all_passed']}   (expected: False, after only {DEMO_EPOCHS} demo epochs)")


---
# Part 4 — Validating against an independent classical solver

## 4.1 The methodology, before any numbers

A trained network that matches a closed-form answer only proves the network learned *that specific algebraic derivation* — it doesn't rule out the possibility that a bug exists in *both* the network's residual code and the closed-form code, if they happened to share the same mistake (e.g. both use the wrong sign convention, or a stale copy-pasted formula). The methodologically stronger check is to bring in a **third, completely independent numerical method** that starts from the *same governing equations* but solves them a *different way*, and see if all three agree.

`src/classical_solver.py` does exactly that: it marches the *differential* form of continuity and Bernoulli's theorem from $x=0$ to $x=L$ using explicit adaptive Runge-Kutta integration (`scipy.integrate.solve_ivp(method="RK45")`) — the same family of numerical method classical CFD texts (e.g. Anderson's *Computational Fluid Dynamics: The Basics with Applications*, Ch. 7) use for quasi-1D nozzle flow. It shares **no code** with `geometry.py` or `analytical.py` — its diameter/area functions are reimplemented independently in plain NumPy, deliberately, so a bug in the PINN's shared geometry module couldn't silently pass a "cross-check" that was secretly checking the PINN against itself.

**What a red flag would have looked like:** if the RK45 march had disagreed with the closed form by more than solver tolerance, that would point to a bug somewhere in translating the physics into code — a sign error in a residual, a mislabeled variable, an inconsistent unit conversion — and it would mean *neither* the PINN's training target nor the "ground truth" it's graded against could be trusted until found. Agreement to near machine precision (which is what we're about to see) is exactly the outcome that lets us trust the rest of this pipeline.

### Honest disclosure: what this is, and what it is NOT

- **It IS** an independent numerical solve of the same governing ODEs, by a genuinely different technique (adaptive RK marching vs. a trained neural network + autodiff). Agreement between them is real evidence the physics was implemented correctly.
- **It is NOT** a meshed, multi-dimensional CFD solve. A real OpenFOAM case for this nozzle would discretize the full 2D/3D Navier–Stokes equations over a volume mesh (viscous terms, wall boundary layers, possible separation) and could disagree with this quasi-1D inviscid model wherever those effects matter. This environment has no OpenFOAM installation or meshing pipeline, so a mesh-based run was not performed. All three methods here — the PINN, the closed form, and this RK45 solver — start from the *same* quasi-1D, inviscid, incompressible model (Part 1). Their agreement proves that model was solved correctly three different ways; it does **not** prove that model is itself a perfect description of the real, viscous, 3D flow in an actual machined nozzle.

## 4.2 The derivation this solver actually integrates

Differentiate continuity and Bernoulli's *integrated* forms (Eqs. 1-2) once more, by hand, to get the coupled ODE system this file marches:

$$\frac{dV}{dx} = -\frac{V}{A(x)}\frac{dA}{dx}(x) \qquad\qquad \frac{dp}{dx} = -\rho\, V\,\frac{dV}{dx}$$

with initial condition $[V(0), p(0)] = [V_{in}, p_{in}]$ — the same inlet condition the PINN hard-wires and the closed form starts its algebra from.

In [ ]:
from src import classical_solver

print(inspect.getsource(classical_solver.d_area_dx))
print(inspect.getsource(classical_solver.solve_nozzle_rk45))


## 4.3 Three-way comparison

We'll compare, for the same throat diameter: (a) the closed-form exact answer, (b) the independent RK45 march, and (c) our short-trained demo PINN from Part 3 (honestly labeled as under-trained — it's here to show the *comparison workflow*, not to claim a passing result the 400-epoch demo was never meant to produce).

In [ ]:
Dt_compare = 0.30  # SI throat diameter [m]
V_in = cfg["physics"]["V_in"]

# (a) closed form
x_grid = np.linspace(0.0, cfg["geometry"]["L"], 201)
xt_grid = torch.as_tensor((x_grid / cfg["geometry"]["L"]).reshape(-1, 1), dtype=torch.float64)
dtt_grid = torch.full_like(xt_grid, Dt_compare / cfg["geometry"]["D_in"])
V_exact = (analytical.velocity_exact_nondim(xt_grid, dtt_grid) * V_in).numpy().flatten()
p_exact = (cfg["physics"]["p_in"] + analytical.pressure_exact_nondim(xt_grid, dtt_grid).numpy().flatten()
           * 0.5 * cfg["physics"]["rho"] * V_in ** 2)

# (b) independent RK45 march
rk45 = classical_solver.solve_nozzle_rk45(Dt_compare, V_in, cfg, n_report=201)

# (c) our short-trained demo PINN
with torch.no_grad():
    V_pinn_nd, p_pinn_nd = demo_model(xt_grid, dtt_grid)
V_pinn = (V_pinn_nd * V_in).numpy().flatten()
p_pinn = (cfg["physics"]["p_in"] + p_pinn_nd.numpy().flatten() * 0.5 * cfg["physics"]["rho"] * V_in ** 2)

rel_l2 = lambda a, b: np.linalg.norm(a - b) / np.linalg.norm(b)
print("Pairwise relative L2 differences, throat diameter =", Dt_compare, "m:")
print(f"  RK45 vs closed form   -- V: {rel_l2(rk45['V'], V_exact):.2e}   p: {rel_l2(rk45['p'], p_exact):.2e}")
print(f"  demo PINN vs closed form -- V: {rel_l2(V_pinn, V_exact):.2e}   p: {rel_l2(p_pinn, p_exact):.2e}")
print("\nRK45 agrees with the closed form to near machine precision -- two independent methods, same answer.")
print("The demo PINN's gap is real and expected: it's had only 400 training epochs, not 20,000+5,000.")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(x_grid, V_exact, "k-", lw=2, label="closed form (Eq. 4)")
axes[0].plot(rk45["x"], rk45["V"], "g--", lw=2, label="independent RK45 march")
axes[0].plot(x_grid, V_pinn, "r:", lw=2, label=f"demo PINN ({DEMO_EPOCHS} epochs)")
axes[1].plot(x_grid, p_exact, "k-", lw=2, label="closed form (Eq. 4)")
axes[1].plot(rk45["x"], rk45["p"], "g--", lw=2, label="independent RK45 march")
axes[1].plot(x_grid, p_pinn, "r:", lw=2, label=f"demo PINN ({DEMO_EPOCHS} epochs)")
axes[0].set(xlabel="x [m]", ylabel="V [m/s]", title=f"Velocity, three methods ($D_t$={Dt_compare} m)")
axes[1].set(xlabel="x [m]", ylabel="p [Pa]", title=f"Pressure, three methods ($D_t$={Dt_compare} m)")
for ax in axes:
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


---
# Part 5 — The inference-speed argument, measured honestly

**The argument, in one sentence:** train once, expensively; then answer unlimited queries almost for free — the payoff grows the more queries you ultimately need, because training is a one-time cost paid regardless of whether you query the model once or a million times afterward, while the classical solver pays its full solve cost again, from scratch, every single query.

Let's measure this, not assert it. `src/benchmark.py` times two things that answer the exact same question by different mechanisms: (1) `classical_solver.solve_nozzle_rk45` — one fresh RK45 integration *per query*, nothing reused between queries; (2) our demo PINN's `forward()` — one batched matrix-multiply pass answering *all* queries in the batch at once.

In [ ]:
print(inspect.getsource(benchmark.time_classical_solver))
print(inspect.getsource(benchmark.time_pinn_inference))


In [ ]:
t0 = time.time()
bench_rows = benchmark.run_benchmark(demo_model, cfg, n_queries_list=[1, 10, 100, 1000])
print(f"(benchmark itself took {time.time()-t0:.1f}s to run)\n")

print(f"{'n_queries':>10} | {'classical total (s)':>20} | {'PINN total (s)':>16} | {'speedup':>10}")
for r in bench_rows:
    print(f"{r['n_queries']:>10} | {r['classical_total_s']:>20.4f} | {r['pinn_total_s']:>16.5f} | {r['speedup_x']:>9.0f}x")


In [ ]:
ns = [r["n_queries"] for r in bench_rows]
speedups = [r["speedup_x"] for r in bench_rows]
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.loglog(ns, speedups, "o-")
ax.set(xlabel="number of queries (log scale)", ylabel="speedup vs classical solver (log scale)",
       title="PINN inference speedup grows with query count")
ax.grid(alpha=0.3, which="both")
plt.show()


**Honest caveat, not a footnote — read this before quoting a speedup number.** This network is tiny (21,122 parameters) and this ran on a shared, CPU-only machine with no GPU. At small query counts, Python-level call overhead and ordinary system noise (other processes, cache state) are a meaningful fraction of the measured time, so the speedup ratio above will **not** scale as a clean, predictable curve the way the one-sentence argument suggests in the idealized limit — it's a real, reproducible measurement on this actual hardware, not a theoretical extrapolation. Also note: the one-time training cost (Part 3, tens of minutes for the full run) is **not** included in any of these per-query numbers above — it's paid once, up front, separately, and is exactly why this argument only pays off once you need *many* queries, not for a single one-off evaluation.

---
# Part 6 — Now build your own

Everything above generalizes. Here's the minimum checklist for applying this exact method to a *different* physics problem, with a pointer back to the specific part of this notebook that demonstrated each step:

1. **Write the governing equations.** Start from first principles (Part 1.3-1.4 derived continuity and Bernoulli from mass conservation and $F=ma$) — know exactly what differential equation(s) your unknown function(s) must satisfy.
2. **Non-dimensionalize.** Rescale every variable to sit around order 1 (Part 1.5) — neural networks train badly across quantities spanning many orders of magnitude.
3. **Decide hard vs. soft boundary conditions.** Part 2.3 showed both: a soft penalty loss term (simple, but a suggestion the optimizer can trade off) vs. an algebraic hard-BC output transform (guaranteed, when the boundary geometry is simple enough to bake in).
4. **Pick a smooth activation function**, and know why smoothness matters: Part 0.2 explained that the residual comes from *differentiating the network's output* — a kinked activation (like ReLU) would poison that derivative. `tanh` or `sin` are the safe defaults.
5. **Write the residual-based loss**: for each governing equation, `residual → square → mean` (Part 2.4, and the toy version in Part 0.4). Start with fixed weights ($\lambda=1$) before reaching for annealing or any other refinement.
6. **Sample collocation points** across your domain (and any design parameters, if you want a parametric surrogate) — remember these are NOT labeled data (Part 2.5); resample periodically so the network can't quietly overfit to one fixed point set.
7. **Train with Adam, then polish with L-BFGS** (Part 3.1-3.2) — and audit any multi-term, order-sensitive logic (like loss annealing) for exactly the kind of "must happen before backward()" bug documented here.
8. **Validate against an independent baseline before trusting anything** (Part 4) — ideally a numerical method that shares no code with your training pipeline, and be explicit about what that comparison does and does not prove.
9. **Benchmark inference speed honestly** (Part 5) — measure the actual amortization curve on your actual hardware rather than asserting a theoretical speedup, and always report the one-time training cost alongside it.

That's the whole method. Every piece of machinery in this notebook — hard boundary conditions, residual losses, collocation sampling, LR-annealing, batched inference — exists to serve one of these nine steps, and none of it is specific to nozzles or to Bernoulli's theorem.